# Лекция 1. Знакомство с обучением с подкреплением

**Цели лекции**

1. Выделить основные отличительные черты Reinforcement Learning.
2. Понимать области применения Reinforcement Learning.
3. Разобраться, что отличает Reinforcement Learning от других областей машинного обучения.

**План**

1. Как устроен курс
2. Что такое Reinforcement Learning
3. Три вида машинного обучения: supervised, unsupervised, reinforcement
4. Где применяется RL
5. Ключевые понятия: состояние, действие, награда, политика, эпизод
6. Примеры сред: Maze, Frozen Lake, Atari, CartPole, Mountain Car, гуманоид
7. Определите сами: состояние, действие, награда в задачах из жизни
8. Живое демо: CartPole в Gymnasium
9. Живое демо: тележка на горке, хорошая и плохая политика
10. Случайные процессы, марковские цепи и MDP
11. Награда как формулировка задачи
12. Исследование и использование (exploration vs exploitation)
13. Первый алгоритм: метод Cross-Entropy
14. Карта курса, итоги, литература

## 1. Как устроен курс

* **16 недель**, одно занятие в неделю: лекция + семинар. Каждая неделя — папка `NN-topic-name/` в репозитории с тремя ноутбуками: `lecture/`, `seminar/`, `homework/`.
* **Домашние задания** почти каждую неделю, в `.ipynb`. Часть проверок — через `assert`, так что можно проверить себя до сдачи.
* **Оценка** (черновик, обсуждаем сегодня): 60% домашние задания, 30% итоговый проект, 10% активность на семинарах.
* **Итоговый проект**: своя реализация RL-агента или мини-исследование на среде по выбору. Темы выбираем на неделе 14, защита на неделе 16.
* **Инструменты**: Python 3.10+, [Gymnasium](https://gymnasium.farama.org/) (среды), [PyTorch](https://pytorch.org/) (нейросети), Jupyter.
  Библиотеки [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) и [CleanRL](https://github.com/vwxyzjn/cleanrl) читаем как справочник, но алгоритмы в домашках пишем сами.
* **Что нужно уметь на входе**: линейная алгебра, теория вероятностей, градиентный спуск, Python с numpy. Нейросети и PyTorch — желательно, но необходимый минимум разберём на мини-семинаре `../seminar/pytorch_intro.ipynb`.

Установка окружения:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

## 2. Что такое Reinforcement Learning

Начнём с картинки, которую все понимают без определений: дрессировщик и собака.

<img src="../../assets/ref/agent_env_dog.png" width="420">

Собака **наблюдает** (жест, голос), **действует** (садится, лает, убегает), а дрессировщик выдаёт **награду** (лакомство есть или нет). Никто не объясняет собаке, *как* правильно, есть только последствия. Через десяток повторений действие, за которым идёт лакомство, случается чаще. Это и есть обучение с подкреплением.

Та же схема в общем виде:

<img src="../../assets/ref/agent_env_brain.png" width="420">

* **Агент** — тот, кто принимает решения (мозг, программа). Единственное, что мы обучаем.
* **Среда** — всё остальное: физический мир, игра, биржа, другой игрок.
* На каждом шаге $t$ агент видит **состояние** $S_t$ (или наблюдение), выбирает **действие** $A_t$, среда отвечает **наградой** $R_t$ и новым состоянием $S_{t+1}$.

**Цель агента** — максимизировать суммарную награду

$$
G = \sum_{t=0}^{\infty} \gamma^t R_t, \qquad \gamma \in [0, 1].
$$

**Reinforcement Learning (RL)** — раздел машинного обучения о том, как агент, взаимодействуя со средой, учится выбирать действия так, чтобы суммарная награда была как можно больше.

<small>Иллюстрации в этом и следующем разделах — из лекции 1 курса [imm-rl-lab](https://github.com/imm-rl-lab/reinforcement_learning_course) (А. Плаксин).</small>

### Откуда слово «подкрепление»

Термин пришёл из психологии. **Закон эффекта** Торндайка (1911): действия с приятными последствиями в той же ситуации повторяются чаще. **Подкрепление** (reinforcement) у Скиннера (1938): всё, что увеличивает частоту поведения. Математику добавила теория оптимального управления: **Беллман** (1957) ввёл ценность состояния и уравнение, которое мы сегодня увидим. **Саттон и Барто** в 1980-х соединили обе линии в то, что теперь называется RL.

![origins](../../assets/rl_origins.png)

## 3. Три вида машинного обучения

Прежде чем углубляться в RL, поймём, где он живёт на карте машинного обучения. Обычно выделяют три большие группы задач, и отличаются они не алгоритмами, а тем, **какие данные нам доступны** и **откуда берётся сигнал для обучения**.

![paradigms](../../assets/ml_paradigms.png)

### Обучение с учителем (supervised learning)

Дана таблица примеров с правильными ответами: пары $(x_i, y_i)$. Модель ищет функцию $f(x) \approx y$, ошибка на каждом примере известна сразу.

* **Спам-фильтр**: $x$ — текст письма, $y$ — «спам / не спам». Разметку сделали люди, нажимая кнопку «в спам».
* **Диагностика по снимку**: $x$ — рентгеновский снимок, $y$ — диагноз врача.
* **Прогноз спроса**: $x$ — день недели, погода, цена, акции конкурентов; $y$ — сколько единиц товара продали. Ответ известен из истории.
* **Перевод текста**: $x$ — предложение по-английски, $y$ — его перевод из параллельного корпуса.

Ключевое: **учитель уже знает правильный ответ** для каждого примера, а сами данные лежат в файле и не меняются от того, что делает модель.

### Обучение без учителя (unsupervised learning)

Ответов нет, есть только объекты $x_i$. Задача — найти в них структуру.

* **Кластеризация клиентов**: разбить покупателей на группы по поведению; никто заранее не знает, сколько групп и какие они.
* **Поиск аномалий**: транзакция не похожа на все остальные — возможно, мошенничество.
* **Снижение размерности и эмбеддинги**: сжать 10 000 признаков в 50, сохранив «похожесть» объектов (PCA, автоэнкодеры, word2vec).
* **Тематическое моделирование**: о чём эти 100 000 новостей, если тем никто не размечал.

Ключевое: **правильного ответа не существует в принципе**, качество оценивается косвенно.

### Обучение с подкреплением (reinforcement learning)

Нет ни таблицы с ответами, ни фиксированного набора данных. Есть **среда**, в которой агент действует, и **награда** — число, которое приходит в ответ на действия.

* **Игра**: агент видит экран, нажимает кнопки, в конце партии узнаёт, выиграл или проиграл.
* **Робот**: подаёт токи на моторы, награда — прошёл ли метр вперёд и не упал ли.
* **Торговый агент**: покупает и продаёт, награда — изменение стоимости портфеля за вычетом издержек.
* **Охлаждение дата-центра**: меняет уставки кондиционеров, награда — минус потраченная энергия, пока температура в норме.

Ключевое: **никто не говорит, какое действие было правильным** — только насколько хорошо получилось в итоге, причём часто сильно позже.

### Сравнение

| | Supervised Learning | Unsupervised Learning | Reinforcement Learning |
|---|---|---|---|
| Что дано | размеченные пары $(x, y)$ | неразмеченные $x$ | среда, с которой можно взаимодействовать |
| Обратная связь | правильный ответ сразу | нет обратной связи | награда: число, часто с задержкой |
| Откуда данные | собраны заранее, фиксированы | собраны заранее, фиксированы | порождает сам агент своими действиями |
| Что оптимизируем | ошибку предсказания | меру структуры (расстояния, правдоподобие) | суммарную награду за эпизод |
| Влияют ли решения модели на данные | нет | нет | **да**: политика определяет, что агент увидит |
| Пример | «кошка или собака на фото?» | «на какие группы делятся клиенты?» | «как пройти уровень?» |

**Четыре отличительные черты RL**

1. **Нет правильного ответа, есть оценка.** В supervised learning учитель говорит «здесь должно быть 7». В RL никто не говорит «здесь нужно было повернуть налево», есть только число.
2. **Отложенная награда.** В шахматах награда приходит через 40 ходов после решающей ошибки. Какое из 40 действий виновато? Это задача **credit assignment**.
3. **Данные порождает сам агент.** Плохая политика видит только плохие состояния. Данные не независимы и меняются по ходу обучения; гарантии статистики для i.i.d. выборок здесь не работают.
4. **Нужно исследовать.** Чтобы найти лучшее поведение, надо пробовать новое, а новое в среднем невыгодно. В supervised learning такой проблемы нет вообще (раздел 12).

Общее с остальным ML — статистика и нейросети как инструмент. Отличается сама **постановка задачи**: не «предскажи», а «действуй».

Границы, конечно, размыты: RL умеет пользоваться и разметкой (imitation learning: учимся повторять за экспертом — это уже supervised), и предобученными без учителя представлениями. А в разделе 13 мы увидим алгоритм, который на каждом шаге превращает задачу RL в обычную задачу обучения с учителем на данных, которые агент собрал себе сам.

### Разминка: какой это тип обучения?

Прочитайте формулировку и решите, что это — supervised, unsupervised или reinforcement learning. Обращайте внимание на два вопроса: *есть ли правильные ответы* и *влияют ли решения модели на будущие данные*.

1. По истории поездок предсказать время в пути из дома в университет.
2. Научить манипулятор перекладывать детали из коробки в коробку; известен только процент успешно переложенных.
3. Разбить 50 000 отзывов на тематические группы, чтобы понять, на что жалуются.
4. По фотографии определить, есть ли на детали трещина; 5 000 фотографий размечены контролёром.
5. Подобрать порядок показа новостей в ленте так, чтобы пользователь читал больше в течение месяца.

<details>
<summary>Ответы</summary>

1. **Supervised** (регрессия): для каждой прошлой поездки известно фактическое время.
2. **Reinforcement**: правильная траектория манипулятора неизвестна, есть только результат попытки, и следующие попытки зависят от текущего поведения.
3. **Unsupervised**: разметки тем нет.
4. **Supervised** (бинарная классификация): есть готовые метки контролёра.
5. **Reinforcement**: правильного порядка не существует, награда отложена (реакция за месяц), и лента сама определяет, какие данные мы увидим завтра — классический пример того, как решения модели меняют распределение данных.

</details>

## 4. Где применяется RL

Общий рецепт: как только у задачи есть **среда** (или её симулятор) и **числовая цель**, поведение можно не программировать, а выучить.

![timeline](../../assets/rl_timeline.png)

### Игры

| Год | Система | Что произошло | Посмотреть |
|---|---|---|---|
| 1992 | **TD-Gammon** | Нейросеть + TD-обучение играет в нарды на уровне чемпионов мира. | [статья](https://dl.acm.org/doi/10.1145/203330.203343) |
| 2015 | **DQN** | Одна сеть учится играть в 49 игр Atari по пикселям и счёту. | [видео Breakout](https://www.youtube.com/watch?v=TmPfTpjtdgg), [Nature](https://www.nature.com/articles/nature14236) |
| 2016 | **AlphaGo** | Победа над Ли Седолем в го. | [фильм](https://www.youtube.com/watch?v=WXuK6gekU1Y) |
| 2017 | **AlphaZero** | Шахматы, сёги и го с нуля, только игрой с самим собой. | [блог](https://deepmind.google/discover/blog/alphazero-shedding-new-light-on-chess-shogi-and-go/) |
| 2019 | **OpenAI Five**, **AlphaStar** | Dota 2 и StarCraft II на уровне профессионалов. | [OpenAI Five](https://openai.com/index/openai-five/), [AlphaStar](https://deepmind.google/discover/blog/alphastar-mastering-the-real-time-strategy-game-starcraft-ii/) |
| 2019 | **Hide and Seek** | Агенты в прятках сами изобретают использование инструментов. | [блог с гифками](https://openai.com/index/emergent-tool-use/) |

Почему игры: есть симулятор, чёткая награда и можно сыграть миллионы партий.

### Робототехника

[Роботы-футболисты DeepMind](https://sites.google.com/view/op3-soccer), [ходьба четвероногих ANYmal](https://arxiv.org/abs/1901.08652), [Boston Dynamics Spot](https://bostondynamics.com/blog/starting-on-the-right-foot-with-reinforcement-learning/), [кубик Рубика одной рукой](https://openai.com/index/solving-rubiks-cube/). Общая схема: учим в симуляторе, переносим на железо (sim-to-real).

### Управление инфраструктурой

* [Охлаждение дата-центров Google](https://deepmind.google/discover/blog/deepmind-ai-reduces-google-data-centre-cooling-bill-by-40/): минус 40% энергии на охлаждение. Состояние — сотни датчиков, действия — уставки оборудования, награда — энергия при соблюдении температурных ограничений.
* [Управление плазмой в токамаке](https://www.nature.com/articles/s41586-021-04301-9): RL-контроллер держит форму плазмы через токи в 19 магнитных катушках.
* Светофоры, балансировка сетей, [размещение блоков на чипе](https://deepmind.google/discover/blog/how-alphachip-transformed-computer-chip-design/).

### Финансы: торговля на бирже

Исполнение крупных ордеров (как разбить заявку, чтобы не сдвинуть цену), маркет-мейкинг, управление портфелем. Честная оговорка: рынок нестационарен и очень шумный, поэтому это гораздо труднее, чем Atari. Обзор: [Deep RL for trading](https://arxiv.org/abs/1911.10107), учебная библиотека [FinRL](https://github.com/AI4Finance-Foundation/FinRL).

### Роевое поведение дронов

Несколько агентов учатся одновременно держать строй, облетать препятствия, вместе искать цель. Каждый дрон видит только соседей, награда общая — это **multi-agent RL** (неделя 15). [Полёт роя сквозь лес](https://arxiv.org/abs/2202.03308), среды [PettingZoo](https://pettingzoo.farama.org/) и [gym-pybullet-drones](https://github.com/utiasDSL/gym-pybullet-drones).

### Рекомендации и языковые модели

Что показать пользователю, чтобы он остался надолго, а не только кликнул сейчас. **RLHF** ([InstructGPT](https://arxiv.org/abs/2203.02155)): ChatGPT стал полезным собеседником благодаря RL на человеческих предпочтениях; **рассуждающие модели** ([DeepSeek-R1](https://arxiv.org/abs/2501.12948)) обучены RL на проверяемых наградах.

**Интерактивные демо**: [ReinforceJS](https://cs.stanford.edu/people/karpathy/reinforcejs/) (GridWorld в браузере), [каталог сред Gymnasium](https://gymnasium.farama.org/environments/classic_control/), [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction).

## 5. Ключевые понятия

Договоримся о словаре, которым будем пользоваться весь курс.

<img src="../../assets/ref/agent_env_brain.png" width="420">

* **Состояние** $S_t$ — описание ситуации в среде в момент $t$: клетка в лабиринте, четыре числа в CartPole, кадр игры. Состояние должно быть достаточным для принятия решения (об этом — марковское свойство в разделе 10).
* **Наблюдение** $O_t$ — то, что реально видит агент. Если агент видит не всё (покер, туман войны в стратегии), состояние и наблюдение различаются; пока считаем, что они совпадают.
* **Действие** $A_t$ — то, чем агент влияет на среду. Действия бывают **дискретные** (4 направления, 18 кнопок джойстика) и **непрерывные** (момент на каждом из 17 суставов гуманоида). Множество допустимых действий может зависеть от состояния: в шахматах доступны только легальные ходы.
* **Награда** $R_t$ — число, приходящее от среды после действия. Это единственный сигнал обучения: агент не знает, какое действие было «правильным», он знает только, сколько ему заплатили.
* **Политика** $\pi$ — правило выбора действия по состоянию, то, что мы обучаем. Детерминированная политика $a = \pi(s)$ выдаёт одно действие; **стохастическая** политика $\pi(a \mid s)$ задаёт распределение над действиями. Стохастичность нужна не для красоты: пока агент учится, он обязан пробовать разное (раздел 12), и именно с таких политик мы начнём в разделе 13.

### Эпизод и терминальное состояние

**Терминальное состояние** — состояние, из которого взаимодействие не продолжается: шест упал, агент дошёл до цели, партия закончилась, робот перевернулся. Попав в него, агент больше не получает наград, и по определению $V(\text{терминальное}) = 0$ — именно поэтому в формулах рекурсия «обрывается» на терминальном состоянии, а в коде мы обнуляем ценность последнего шага.

**Эпизод** — одна траектория от начального состояния до терминального: одна партия, один заезд, одна попытка робота. Задачи бывают:

* **эпизодические** — игра, лабиринт, посадка корабля: есть естественный конец, после которого среда сбрасывается (`env.reset()`);
* **непрерывные** — управление котельной, торговый агент, рекомендации: конца нет, поток решений бесконечен.

В Gymnasium шаг возвращает **два разных флага**, и это важно не путать:

* `terminated=True` — среда действительно пришла в терминальное состояние (цель достигнута, агент погиб). Будущих наград нет;
* `truncated=True` — эпизод оборвали снаружи по лимиту шагов (в CartPole это 500 шагов). Среда могла бы продолжаться, поэтому обнулять ценность здесь **некорректно** — на неделе 3 мы увидим, как эта разница влияет на обучение.

**Траектория** $\tau = (S_0, A_0, R_0, S_1, A_1, R_1, \ldots)$ — то, что получилось за эпизод. Качество траектории измеряется **суммарной дисконтированной наградой** (return):

$$
G_t = R_t + \gamma R_{t+1} + \gamma^2 R_{t+2} + \ldots = \sum_{k \ge 0} \gamma^k R_{t+k}, \qquad \gamma \in [0, 1].
$$

Коэффициент $\gamma$ говорит, насколько агента волнует далёкое будущее: награда через $k$ шагов весит $\gamma^k$, а «горизонт планирования» примерно $1/(1-\gamma)$ шагов. При $\gamma = 0$ агент близорук и хватает то, что дают сейчас; при $\gamma \to 1$ он готов терпеть ради выигрыша в конце. Для эпизодических задач с коротким эпизодом можно брать $\gamma = 1$.

**Цель RL**: найти политику $\pi$, максимизирующую ожидаемый return $\mathbb{E}_\pi[G_0]$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

ks = np.arange(0, 200)
for gamma in [0.5, 0.9, 0.99]:
    plt.plot(ks, gamma ** ks, label=f"γ = {gamma}: горизонт ≈ {1/(1-gamma):.0f} шагов")
plt.xlabel("k (через сколько шагов придёт награда)")
plt.ylabel("вес награды γ^k")
plt.title("Дисконтирование: насколько агенту важно будущее")
plt.legend()
plt.show()

## 6. Примеры сред

Для каждой задачи нужно ответить на три вопроса: **что агент видит** (состояния), **что он может делать** (действия), **за что ему платят** (награда). Начнём с самого простого.

### Maze

<img src="../../assets/ref/maze.png" width="300">

* **Состояния**: белые клетки лабиринта. Агент всегда знает, в какой клетке он стоит.
* **Действия**: ↑, →, ↓, ←. Шаг в стену оставляет агента на месте.
* **Награда**: −1 на каждом шаге, 0 если стоять в Goal. Чем быстрее дошёл, тем меньше штрафа.
* **Эпизод**: от Start до Goal (или до лимита шагов).

Обратите внимание: агенту никто не говорит «иди направо». Он узнаёт, что путь хороший, только по сумме штрафов в конце. Это то самое отличие RL от обучения с учителем, о котором шла речь в разделе 3.

### Frozen Lake

<img src="../../assets/ref/frozen_lake.png" width="520">

* **Состояния**: 16 клеток озера 4×4. `S` старт, `F` лёд, `H` прорубь, `G` цель.
* **Действия**: влево, вниз, вправо, вверх.
* **Награда**: +1 за достижение `G`, 0 во всех остальных случаях. Попал в прорубь — эпизод окончен без награды.
* **Особенность**: лёд скользкий. Агент идёт туда, куда хотел, лишь с вероятностью 1/3, иначе его сносит вбок. Одно и то же действие в одном и том же состоянии может привести в разные клетки. Это первый пример **случайной среды**.

С Frozen Lake будем работать на семинаре.

### Atari Games

<img src="../../assets/ref/atari.png" width="300">

* **Состояния**: пиксели с экрана (210×160×3 чисел). Агент видит то же, что и человек.
* **Действия**: →, ←, «0» (ничего не делать), огонь и их комбинации, всего до 18.
* **Награда**: очки в игре.

В 2015 году одна и та же нейросеть (DQN, неделя 6) научилась играть в 49 игр Atari, глядя только на пиксели и счёт. Это событие сделало RL знаменитым.

### CartPole

<img src="../../assets/ref/cartpole.png" width="380">

* **Состояния**: $\mathbb{R}^4$ — положение и скорость тележки, угол и угловая скорость шеста. Или пиксели с экрана.
* **Действия**: толкнуть тележку → или ←.
* **Награда**: +1 на каждом шаге, пока шест стоит; эпизод заканчивается, когда шест упал или тележка уехала за край.

Задача простая, но в ней впервые появляется **непрерывное пространство состояний**: таблицу «состояние → действие» уже не составить. Через минуту запустим её вживую.

### Mountain Car

Машинка стоит в низине между двумя горками, мотор слабый: «в лоб» на правый холм заехать не получится. Чтобы добраться до флажка, нужно сначала откатиться назад и раскачаться.

* **Состояние**: два числа — позиция $x \in [-1.2,\, 0.6]$ и скорость $v \in [-0.07,\, 0.07]$. Оба непрерывные.
* **Действия**: три дискретных — газ влево, ничего, газ вправо.
* **Награда**: $-1$ на каждом шаге. Никаких подсказок «ты едешь в правильную сторону» нет.
* **Терминальное состояние**: позиция $\ge 0.5$, машинка на вершине справа. Эпизод также обрывается по лимиту шагов (`truncated`).

Эта среда — отличная иллюстрация двух вещей сразу. Во-первых, **жадность к мгновенной награде вредна**: чтобы в итоге получить меньше штрафа, надо сначала поехать *от* цели. Во-вторых, награда здесь **разрежённая** (sparse): пока агент ни разу не доехал, все траектории одинаково плохи, и учиться буквально не на чем. Мы увидим это своими глазами в разделах 9 и 13.

### Гуманоид (MuJoCo)

<img src="../../assets/ref/humanoid.png" width="300">

* **Состояния**: $\mathbb{R}^{26}$ и больше — углы и скорости всех суставов, положение центра масс.
* **Действия**: $\mathbb{R}^{6}$ … $\mathbb{R}^{17}$ — усилия в каждом суставе. **Действия непрерывные**: не «влево / вправо», а число.
* **Награда**: +1 за каждый момент времени, пока робот не упал, плюс бонус за скорость движения вперёд.

Такими задачами занимаются методы для непрерывного управления (недели 10 и дальше). Их же используют, чтобы учить ходить настоящих четвероногих и двуногих роботов.

## 7. Определите сами

Умение **сформулировать** задачу на языке RL важнее знания конкретных алгоритмов: неправильно выбранная награда или состояние испортят любой метод. Потренируемся. Для каждой задачи назовите состояние, действия, награду, эпизод и найдите, где в среде случайность. Ответы спрятаны, сначала подумайте сами.

### Шахматы

<details>
<summary>Ответ</summary>

* **Состояние**: расположение фигур на доске, чей ход, право на рокировку. Всё видно, скрытой информации нет.
* **Действия**: любой допустимый ход. Множество действий зависит от состояния.
* **Награда**: +1 победа, −1 поражение, 0 ничья, всё остальное время 0. Награда приходит **только в конце**, через десятки ходов.
* **Эпизод**: партия.
* **Случайность**: в правилах её нет, но есть соперник. С точки зрения агента ход соперника — это случайный отклик среды.
</details>

### Такси (Gymnasium `Taxi-v3`)

Такси ездит по сетке 5×5, нужно забрать пассажира в одной из 4 точек и отвезти в другую.

<details>
<summary>Ответ</summary>

* **Состояние**: позиция такси (25 вариантов) × где пассажир (4 точки или в машине) × куда ехать (4 точки) = 500 состояний.
* **Действия**: 4 направления, посадить, высадить.
* **Награда**: −1 за каждый шаг, +20 за успешную высадку, −10 за попытку посадить/высадить не там.
* **Эпизод**: от появления пассажира до высадки.
* **Случайность**: только в начальном состоянии (где появится пассажир и куда ему нужно). Сами переходы детерминированные.
</details>

### Охлаждение дата-центра

Нужно управлять насосами, чиллерами и вентиляторами так, чтобы серверы не перегревались, а счёт за электричество был минимальным.

<details>
<summary>Ответ</summary>

* **Состояние**: показания сотен датчиков: температуры, нагрузка серверов, погода снаружи, текущие уставки оборудования. Агент видит не всё (например, не знает нагрузку через час), это **наблюдение**, а не полное состояние.
* **Действия**: уставки оборудования: обороты насосов, температура воды. Действия **непрерывные**.
* **Награда**: минус потреблённая энергия за шаг, большой штраф за выход температуры за допустимый диапазон.
* **Эпизод**: задача **непрерывная**, естественного конца нет. Удобно резать на дни.
* **Случайность**: погода, нагрузка на серверы, износ оборудования.
</details>

### Торговый агент

Программа управляет портфелем из нескольких акций.

<details>
<summary>Ответ</summary>

* **Состояние**: история цен и объёмов, текущая позиция, остаток денег. Настоящего «состояния рынка» агент не знает, это снова наблюдение.
* **Действия**: купить / продать / держать по каждой бумаге, или доля портфеля в каждой.
* **Награда**: изменение стоимости портфеля за шаг с учётом комиссий; часто ещё штраф за риск.
* **Эпизод**: торговый день или месяц.
* **Случайность**: почти всё. Отклик среды (цена завтра) зависит от миллионов других участников, и правила меняются со временем. Поэтому это одна из самых трудных задач для RL.
</details>

Шпаргалка, как узнать термины в новой задаче:

| Термин | Вопрос, который нужно задать |
|---|---|
| Состояние / наблюдение | Что агент знает в момент принятия решения? Чего он не знает? |
| Действие | Что агент может изменить прямо сейчас? Дискретный выбор или число? |
| Награда | Какое **одно число** за шаг измеряет успех? Приходит сразу или в конце? |
| Эпизод | Есть ли естественный конец? Или задача бесконечная? |
| Случайность среды | Может ли одно и то же действие в одном и том же состоянии привести к разным исходам? |

## 8. Живое демо: CartPole в Gymnasium

[Gymnasium](https://gymnasium.farama.org/) — стандартный интерфейс к средам: `env.reset()` возвращает первое наблюдение, `env.step(action)` — следующее наблюдение, награду и флаги окончания эпизода. Все среды курса будут выглядеть так.

Запустим CartPole.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("observation_space:", env.observation_space)
print("action_space:     ", env.action_space)
print("первое наблюдение:", obs)

obs, reward, terminated, truncated, info = env.step(1)  # 1 = толкнуть вправо
print("после шага:        ", obs, "reward =", reward, "done =", terminated or truncated)

In [ ]:
# Эпизод со случайной политикой: сохраним кадры.
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env.reset(seed=1)
frames = []
for t in range(60):
    frames.append(env.render())
    obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
    if terminated or truncated:
        break
env.close()

idx = np.linspace(0, len(frames) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(16, 2.6))
for ax, i in zip(axes, idx):
    ax.imshow(frames[i])
    ax.set_title(f"t = {i}")
    ax.axis("off")
plt.suptitle(f"Случайная политика: шест падает за {len(frames)} шагов")
plt.show()

Случайная политика держит шест около 20 шагов. Правило выбора действия по состоянию называется **политикой**. Напишем политику руками: если шест падает вправо (угловая скорость положительная), толкаем тележку вправо, и наоборот.

In [ ]:
def random_policy(obs):
    return np.random.randint(2)

def heuristic_policy(obs):
    x, x_dot, theta, theta_dot = obs
    return int(theta_dot > 0)   # толкаем в ту сторону, куда падает шест

def run_episodes(policy, n_episodes=50, seed=0):
    env = gym.make("CartPole-v1")
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(policy(obs))
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return np.array(returns)

np.random.seed(0)
for name, policy in [("случайная", random_policy), ("эвристика", heuristic_policy)]:
    G = run_episodes(policy)
    print(f"{name:10s}: средняя суммарная награда {G.mean():6.1f} ± {G.std():5.1f} (максимум 500)")

Эвристика в 10 раз лучше случайной, но её придумал человек, зная физику задачи, и до максимума она не дотягивает. Для Atari, шахмат или гуманоида с 17 суставами такое правило руками не напишешь.

**Задача RL** — получить политику не хуже эвристики, **не зная** устройства среды, только из опыта взаимодействия. Первый такой алгоритм увидим сегодня в разделе 13, а на неделе 6 нейросеть будет стабильно держать 500 шагов.

Если установлен `box2d`, можно посмотреть среду посложнее: LunarLander.

In [ ]:
try:
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, _ = env.reset(seed=0)
    frames = []
    for t in range(120):
        frames.append(env.render())
        obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
        if terminated or truncated:
            break
    env.close()
    idx = np.linspace(0, len(frames) - 1, 4).astype(int)
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, i in zip(axes, idx):
        ax.imshow(frames[i]); ax.set_title(f"t = {i}"); ax.axis("off")
    plt.suptitle("LunarLander, случайная политика: наблюдение из 8 чисел, 4 действия (двигатели)")
    plt.show()
except Exception as e:  # box2d не установлен или не собрался
    print("LunarLander недоступен:", type(e).__name__, "-", str(e)[:120])
    print("Гифку можно посмотреть в документации: https://gymnasium.farama.org/environments/box2d/lunar_lander/")

## 9. Живое демо: тележка на горке, хорошая и плохая политика

Среда одна, агент один, награда одна — меняется только **политика**. Посмотрим на двух «агентов», написанных руками:

* **«газ вправо»** — жадная к цели политика: цель справа, значит, всё время газуем вправо;
* **«раскачка»** — газуем туда, куда уже едем: `вправо`, если скорость положительна, и `влево`, если отрицательна.

Заодно сделаем маленькую утилиту `show_frames`, которая превращает кадры среды в анимацию прямо в ноутбуке (её же используем в разделе 13).

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

def show_frames(frames, title="", interval=60, width=3.2):
    """Кадры среды -> анимация в ноутбуке (работает при запуске, в превью на GitHub не видна)."""
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.axis("off"); ax.set_title(title, fontsize=10)
    im = ax.imshow(frames[0])
    plt.close(fig)
    anim = animation.FuncAnimation(fig, lambda i: (im.set_data(frames[i]),),
                                   frames=len(frames), interval=interval, blit=True)
    return HTML(anim.to_jshtml(default_mode="loop"))

In [ ]:
def run_mountain_car(policy, seed=0, max_steps=200, every=4):
    """Один эпизод MountainCar. Возвращает кадры (каждый every-й), позиции и флаг успеха."""
    env = gym.make("MountainCar-v0", render_mode="rgb_array", max_episode_steps=max_steps)
    obs, _ = env.reset(seed=seed)
    frames, xs, t, terminated = [], [obs[0]], 0, False
    while True:
        if t % every == 0:
            frames.append(env.render())
        obs, r, terminated, truncated, _ = env.step(policy(obs))
        xs.append(obs[0]); t += 1
        if terminated or truncated:
            break
    env.close()
    return frames, np.array(xs), terminated

gas_right   = lambda obs: 2                      # всегда газ вправо
swing       = lambda obs: 2 if obs[1] > 0 else 0  # газ в ту сторону, куда уже едем

for name, policy in [("«газ вправо»", gas_right), ("«раскачка»", swing)]:
    frames, xs, ok = run_mountain_car(policy)
    print(f"{name:14s}: шагов {len(xs)-1:3d}, максимальная высота x = {xs.max():+.2f}, "
          f"доехал до флажка: {'да' if ok else 'НЕТ'}")
    plt.plot(xs, label=name)

plt.axhline(0.5, color="k", ls="--", lw=1, label="флажок (x = 0.5)")
plt.xlabel("шаг эпизода"); plt.ylabel("позиция x"); plt.legend()
plt.title("Одна среда, две политики"); plt.show()

In [ ]:
frames_bad, _, _ = run_mountain_car(gas_right)
frames_good, _, _ = run_mountain_car(swing)
display(show_frames(frames_bad, "Плохая политика: газ вправо, машинка не выезжает"))
display(show_frames(frames_good, "Хорошая политика: раскачка, машинка доезжает"))

Разница между «доехал» и «не доехал» — не в среде и не в награде, а только в политике. При этом «плохая» политика выглядит совершенно разумно: она всё время едет **к цели**. Проигрывает она потому, что жадничает — не готова временно ехать в противоположную сторону ради выигрыша потом. Ровно этому — терпеть сейчас ради суммарной награды — и учится RL-агент.

Обратите внимание и на другое: политику «раскачка» придумал человек, глядя на физику задачи. В разделе 13 такую же (и лучше) политику найдёт алгоритм, который про горки, тележки и энергию не знает ничего.

## 10. Случайные процессы, марковские цепи и MDP

Чтобы что-то доказывать и программировать, нужна математическая модель среды. Построим её в три шага: марковская цепь → цепь с наградами → процесс принятия решений.

### 10.1 Случайный процесс и марковское свойство

**Случайный процесс** — последовательность случайных величин $S_0, S_1, S_2, \ldots$: состояние среды в моменты времени $0, 1, 2, \ldots$. Погода по дням, курс акции по минутам, клетка, в которой стоит агент.

В общем случае $S_{t+1}$ может зависеть от всей истории. **Марковское свойство** — упрощение, на котором держится весь RL:

$$
\mathbb{P}[S_{t+1} \mid S_t] = \mathbb{P}[S_{t+1} \mid S_0, S_1, \ldots, S_t].
$$

«Будущее зависит от прошлого только через настоящее». Если состояние выбрано правильно, вся полезная история уже в нём: в CartPole поэтому в наблюдении есть скорости, а в Atari складывают 4 последних кадра.

**Марковская цепь** — набор состояний $\mathcal{S}$ и матрица переходов $P$, где $P_{ss'} = \mathbb{P}[S_{t+1} = s' \mid S_t = s]$. Строки матрицы суммируются в 1.

Пример: день студента.

![chain](../../assets/markov_chain.png)

In [ ]:
states = ["Лекция", "Соцсети", "Сон", "Экзамен сдан"]
P = np.array([
    [0.0, 0.5, 0.2, 0.3],   # из Лекции
    [0.3, 0.0, 0.7, 0.0],   # из Соцсетей
    [1.0, 0.0, 0.0, 0.0],   # из Сна
    [0.0, 0.0, 0.0, 1.0],   # Экзамен сдан: терминальное, остаёмся навсегда
])
assert np.allclose(P.sum(axis=1), 1.0)

rng = np.random.default_rng(0)

def sample_chain(P, s0=0, max_steps=20):
    s, path = s0, [s0]
    for _ in range(max_steps):
        s = rng.choice(len(P), p=P[s])
        path.append(s)
        if P[s, s] == 1.0:      # терминальное состояние
            break
    return path

for _ in range(4):
    print(" -> ".join(states[s] for s in sample_chain(P)))

# Распределение через n шагов: p_n = p_0 P^n
p = np.array([1.0, 0, 0, 0])
for n in [1, 2, 5, 20]:
    print(f"через {n:2d} шагов: " + ", ".join(f"{states[i]} {x:.2f}" for i, x in enumerate(p @ np.linalg.matrix_power(P, n))))

Матрица переходов отвечает на любой вопрос о будущем цепи: где будем через $n$ шагов ($p_0 P^n$), с какой вероятностью когда-нибудь сдадим экзамен, сколько в среднем это займёт. Никакой истории хранить не нужно.

### 10.2 Марковский процесс с наградами (MRP)

Добавим к цепи **награду** $R(s)$ за посещение состояния и коэффициент дисконтирования $\gamma$. Теперь у каждой траектории есть **return** — суммарная дисконтированная награда с момента $t$:

$$
G_t = R_{t} + \gamma R_{t+1} + \gamma^2 R_{t+2} + \ldots = \sum_{k=0}^{\infty} \gamma^k R_{t+k}.
$$

Зачем $\gamma < 1$:

* сумма конечна даже для бесконечных траекторий;
* награда через $k$ шагов весит $\gamma^k$: «горизонт планирования» агента $\approx 1/(1-\gamma)$;
* математически удобно (сжимающее отображение, увидим на неделе 3).

**Ценность состояния** — ожидаемый return, если стартовать из $s$:

$$
V(s) = \mathbb{E}[G_t \mid S_t = s].
$$

Ключевое наблюдение: return **рекурсивен**, $G_t = R_t + \gamma G_{t+1}$. Взяв матожидание, получаем **уравнение Беллмана** для MRP:

$$
V(s) = R(s) + \gamma \sum_{s'} P_{ss'} V(s').
$$

Ценность состояния = награда прямо сейчас + дисконтированная средняя ценность того, куда мы попадём. Считать её можно совсем просто: взять любые начальные значения и повторять правую часть, пока числа не перестанут меняться. Проверим результат честной симуляцией: сыграем цепь много раз и усредним returns.

Наградим студента так: −2 за лекцию (скучно), +1 за соцсети (приятно), 0 за сон, +20 за сданный экзамен.

In [ ]:
R = np.array([-2.0, 1.0, 0.0, 20.0])
gamma = 0.9
P_ = P.copy(); P_[3] = 0.0          # после экзамена награды больше нет: терминальное состояние

V = np.zeros(4)
for it in range(500):
    V_new = R + gamma * P_ @ V
    if np.max(np.abs(V_new - V)) < 1e-8:
        break
    V = V_new
for s, v in zip(states, V):
    print(f"V({s:12s}) = {v:7.3f}")
print(f"итераций до сходимости: {it}")

def sample_return(s0, gamma, max_steps=300):
    s, G, k = s0, 0.0, 0
    for _ in range(max_steps):
        G += (gamma ** k) * R[s]
        if s == 3:                   # терминальное состояние
            break
        s = rng.choice(4, p=P[s]); k += 1
    return G

for s0 in [0, 1]:
    mc = np.mean([sample_return(s0, gamma) for _ in range(20_000)])
    print(f"проверка симуляцией: V({states[s0]}) ≈ {mc:6.3f}   (уравнение Беллмана: {V[s0]:6.3f})")

Лекция сама по себе неприятна ($R = -2$), но $V(\text{Лекция})$ выше, чем $V(\text{Соцсети})$: из лекции ближе к экзамену. Ценность учитывает **будущее**, а не только текущую награду. Именно ценность, а не награду, будет максимизировать агент.

### 10.3 MDP: добавляем действия

До сих пор состояния менялись сами. Дадим агенту возможность **влиять** на переходы. **Марковский процесс принятия решений** (Markov Decision Process) — кортеж $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$:

* $\mathcal{S}$ — множество состояний;
* $\mathcal{A}$ — множество действий;
* $P(s' \mid s, a) = \mathbb{P}[S_{t+1} = s' \mid S_t = s, A_t = a]$ — вероятности переходов, теперь они зависят от действия;
* $R(s, a)$ — ожидаемая награда за действие $a$ в состоянии $s$;
* $\gamma$ — коэффициент дисконтирования.

Марковское свойство теперь читается так: $\mathbb{P}[S_{t+1} \mid S_t, A_t] = \mathbb{P}[S_{t+1} \mid S_0, A_0, \ldots, S_t, A_t]$.

**Политика** $\pi(a \mid s)$ — распределение над действиями в состоянии $s$. Детерминированная политика $\pi: \mathcal{S} \to \mathcal{A}$ — частный случай. Как порождается траектория:

* агент в $S_0$, выбирает $A_0 \sim \pi(\cdot \mid S_0)$;
* получает $R_0 = R(S_0, A_0)$ и переходит в $S_1 \sim P(\cdot \mid S_0, A_0)$;
* выбирает $A_1 \sim \pi(\cdot \mid S_1)$, и так далее.

$$
\tau = (S_0, A_0, S_1, A_1, S_2, A_2, \ldots), \qquad G(\tau) = \sum_{t=0}^{\infty} \gamma^t R(S_t, A_t).
$$

Важный факт: **MDP + фиксированная политика = MRP**. Если политика выбрана, действия можно «усреднить» и получить обычную цепь с матрицей $P^\pi_{ss'} = \sum_a \pi(a \mid s) P(s' \mid s, a)$. Значит, всё, что мы умеем для MRP, переносится.

Пример MDP — GridWorld: состояние — клетка, действие — направление, переходы «скользкие» (это и есть стохастичность среды), награда за выход и за яму.

![gridworld](../../assets/mdp_gridworld.png)

### 10.4 Ценности и уравнения Беллмана для MDP

* **Value function** $V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$: насколько хорошо находиться в $s$, следуя $\pi$.
* **Action-value function** $Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$: насколько хорошо в $s$ сделать $a$, а дальше следовать $\pi$.

Из рекурсии $G_t = R_t + \gamma G_{t+1}$ получаются **уравнения Беллмана (ожидания)**:

$$
V^\pi(s) = \sum_a \pi(a \mid s) \Big[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a)\, V^\pi(s') \Big],
$$

$$
Q^\pi(s, a) = R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \sum_{a'} \pi(a' \mid s')\, Q^\pi(s', a').
$$

Связь между ними: $V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a)$.

Для **оптимальной** политики сумма по $a'$ с весами $\pi$ превращается в $\max_{a'}$ — это уравнения оптимальности Беллмана, ими займёмся на неделе 3. Почти все алгоритмы курса так или иначе решают одно из этих уравнений: динамическое программирование — итерациями, TD-обучение — по сэмплам, deep RL — нейросетью, минимизируя невязку.

Маленькая проверка: цепочка из трёх состояний, политика всегда идёт вправо, переход детерминированный, +1 за приход в терминальное $s_2$. Тогда $V^\pi(s_1) = 1$, $V^\pi(s_0) = \gamma$.

In [ ]:
gamma = 0.9
P_pi = np.array([[0, 1, 0],
                 [0, 0, 1],
                 [0, 0, 1]], dtype=float)   # переходы при политике «вправо»
R_pi = np.array([0.0, 1.0, 0.0])           # ожидаемая награда за шаг из s

V = np.zeros(3)
for it in range(30):
    V_new = R_pi + gamma * P_pi @ V
    V_new[2] = 0.0              # терминальное состояние
    if np.max(np.abs(V_new - V)) < 1e-10:
        break
    V = V_new
print(f"сошлось за {it} итераций: V = {V.round(4)}  (ожидаем [γ, 1, 0] = [{gamma}, 1, 0])")

## 11. Награда — это формулировка задачи, и её легко испортить

Агент оптимизирует ровно то, что написано в награде, а не то, что вы имели в виду. Классический пример — [CoastRunners](https://openai.com/index/faulty-reward-functions/): агента-лодку обучали на игровые очки, и вместо прохождения трассы он нашёл лагуну, где можно бесконечно крутиться, собирать бонусы и врезаться в стены. Очков больше, чем у любого честного игрока. Это называется **reward hacking**; обзор с десятками примеров есть у [Lilian Weng](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/).

Практическое правило: награда должна описывать *что* нужно получить, а не *как* это делать. Если написать в награду «держи угол шеста маленьким», агент найдёт способ держать угол маленьким, необязательно тот, который вы ожидали.

## 12. Исследование и использование

Есть дилемма, которой нет ни в supervised, ни в unsupervised learning, и которая всплывёт у нас в каждом алгоритме курса.

* **Использование (exploitation)** — делать то, что по накопленному опыту кажется лучшим. Выгодно **сейчас**.
* **Исследование (exploration)** — пробовать то, про что мы мало знаем. Стоит награды сейчас, окупается **потом**.

Бытовые примеры:

* обед: пойти в проверенное кафе или попробовать новое?
* дорога домой: знакомый маршрут или тот, что предлагает навигатор?
* выбор курса, книги, вида спорта — то же самое.

Проблема в том, что оценка «лучшего» строится по нашему же опыту, а опыт мы собираем сами. Если после одного удачного обеда всегда ходить в одно кафе, мы никогда не узнаем, что через дорогу лучше: наша собственная политика закрыла нам доступ к этим данным. Посмотрим на это численно.

In [ ]:
rng = np.random.default_rng(0)
cafes = {"Столовая": 0.3, "Пекарня": 0.5, "Новое кафе": 0.7}   # вероятность удачного обеда
names, p_true = list(cafes), np.array(list(cafes.values()))
best = int(np.argmax(p_true))

def student(days=100, n_try=1):
    """Сначала по n_try раз пробуем каждое кафе, дальше всегда ходим в лучшее по опыту."""
    wins, visits = np.zeros(3), np.zeros(3)
    happy = 0.0
    for a in range(3):
        for _ in range(n_try):
            r = rng.binomial(1, p_true[a]); wins[a] += r; visits[a] += 1; happy += r
    favourite = int(np.argmax(wins / visits))
    for _ in range(days - 3 * n_try):
        happy += rng.binomial(1, p_true[favourite])
    return favourite, happy / days

for n_try in [1, 3, 10]:
    runs = [student(n_try=n_try) for _ in range(2000)]
    stuck = np.mean([f != best for f, _ in runs])
    happy = np.mean([h for _, h in runs])
    print(f"по {n_try:2d} проб(ы) на кафе: выбрали не лучшее кафе в {stuck:4.0%} случаев, "
          f"довольных обедов {happy:.0%} (максимум {p_true[best]:.0%})")

Чем меньше агент исследовал, тем чаще он навсегда «залипает» на неоптимальном варианте — и не потому, что плохо считает, а потому что перестал собирать данные. Отсюда простое правило, которым мы будем пользоваться весь курс: **политика во время обучения должна оставаться случайной**. Как именно дозировать случайность — тема семинара (там появятся стратегии $\varepsilon$-greedy и UCB1) и следующих недель. Нам сейчас важен сам факт: алгоритм из следующего раздела работает именно потому, что его политика стохастическая.

## 13. Первый алгоритм: метод Cross-Entropy

У нас есть всё, чтобы написать настоящий RL-алгоритм: среда, политика, эпизоды и return.

Казалось бы, можно просто перебрать политики и выбрать лучшую. Посчитаем, сколько их: на Frozen Lake 4×4 состояний 16, действий 4, значит, детерминированных политик $4^{16} \approx 4.3 \cdot 10^9$. На поле 8×8 — уже $4^{64} \approx 3 \cdot 10^{38}$. Перебор безнадёжен, и это ещё простейшие среды.

Значит, политику нужно не перебирать, а **постепенно улучшать по собственному опыту**. Метод Cross-Entropy — самый простой способ это сделать, всего из трёх шагов.

### 13.1 Идея

Представьте, что вы учитесь готовить блюдо без рецепта. Вы готовите его 100 раз, каждый раз чуть по-разному. Потом отбираете 20 самых удачных попыток, смотрите, что в них общего («на этом шаге я обычно убавлял огонь»), и в следующей сотне попыток чаще делаете именно так. Через несколько кругов получается рецепт.

Ровно это и делает Cross-Entropy Method (CEM):

1. **Сыграть.** Провести $K$ эпизодов текущей (случайной) политикой и записать, что в них происходило и чем закончилось.
2. **Отобрать элиту.** Оставить лучшие эпизоды — те, у которых return выше порога.
3. **Подстроить политику.** Сделать так, чтобы действия из элитных эпизодов выбирались чаще.

И повторять. Алгоритм ничего не знает про устройство среды: ни $P(s' \mid s, a)$, ни $R(s, a)$ ему не нужны, он только играет и смотрит на результат — это **model-free** метод. И он не использует уравнение Беллмана: сравниваются целые траектории, а не отдельные шаги. Такие методы называют **эволюционными** или **методами прямого поиска в пространстве политик**.

### 13.2 Политика — это таблица

Пока состояний и действий конечное число, политику удобно хранить матрицей:

$$
\pi[s, a] = \mathbb{P}[A_t = a \mid S_t = s], \qquad \sum_a \pi[s, a] = 1 \ \ \text{для каждого } s.
$$

Строка — состояние, столбец — действие, в клетке вероятность. Начинаем с **равномерной** политики: $\pi[s, a] = 1/|\mathcal{A}|$, то есть агент в каждом состоянии бросает кубик. Как мы выяснили в разделе 12, случайность здесь не недостаток, а необходимое условие: только так среди сотни эпизодов найдутся удачные, у которых есть чему учиться.

Возьмём Frozen Lake 4×4 без скольжения: 16 состояний, 4 действия, награда 1 за достижение цели и 0 во всех остальных случаях.

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
n_states, n_actions = env.observation_space.n, env.action_space.n
print("состояний:", n_states, " действий:", n_actions)
print("карта озера (S — старт, F — лёд, H — прорубь, G — цель):")
print("\n".join("  " + "".join(c.decode() for c in row) for row in env.unwrapped.desc))

def run_session(env, policy, rng, max_steps=100):
    """Один эпизод: возвращает посещённые состояния, выбранные действия и суммарную награду."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))   # действие ~ pi(.|s)
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total

rng = np.random.default_rng(0)
uniform_policy = np.ones((n_states, n_actions)) / n_actions
states_, actions_, G_ = run_session(env, uniform_policy, rng)
print("\nпример эпизода равномерной политики:")
print("  состояния:", states_)
print("  действия: ", actions_, "  (0=←, 1=↓, 2=→, 3=↑)")
print("  return:   ", G_)

### 13.3 Шаг 1: оценить политику (Монте-Карло)

Насколько хороша политика? Честный ответ — $\mathbb{E}_\pi[G]$, но матожидание мы считать не умеем: модели среды нет. Зато мы умеем играть. По закону больших чисел

$$
\mathbb{E}_{\pi}[G] \approx \frac{1}{K} \sum_{k=1}^{K} G(\tau_k),
$$

где $\tau_k$ — сыгранные эпизоды. Это **метод Монте-Карло**, он же **policy evaluation** в простейшем виде. Посмотрим на распределение returns у равномерной политики.

In [ ]:
sessions = [run_session(env, uniform_policy, rng) for _ in range(500)]
returns = np.array([G for _, _, G in sessions])
print(f"500 эпизодов равномерной политики: средний return {returns.mean():.3f}, "
      f"успешных эпизодов {int(returns.sum())}")
plt.hist(returns, bins=[-0.25, 0.25, 0.75, 1.25], rwidth=0.5)
plt.xticks([0, 1], ["провал (0)", "дошёл до цели (1)"])
plt.ylabel("число эпизодов"); plt.title("Из чего алгоритму предстоит извлечь сигнал")
plt.show()

Почти все эпизоды заканчиваются нулём, и лишь около 2% — единицей: случайно добрести до цели удаётся редко. Вот эти полтора десятка удачных эпизодов и есть всё, что у нас есть. Задача — «выжать» из них политику.

### 13.4 Шаг 2: отобрать элиту

Порог не задаём руками, а берём **квантиль** уровня $q$ от распределения returns:

$$
\text{threshold} = \text{quantile}\big(\{G(\tau_k)\}_{k=1}^{K},\ q\big), \qquad
\text{элита} = \{\tau_k : G(\tau_k) \ge \text{threshold}\}.
$$

Почему квантиль, а не фиксированное число: порог **сам подстраивается** под текущий уровень агента. Сегодня элита — это «хоть раз дошёл», через десять итераций — «дошёл за 6 шагов». Планка растёт вместе с политикой, и алгоритму не нужно знать заранее, какая награда в этой среде «хорошая».

Параметр $q$ — доля отбрасываемых эпизодов: при $q = 0.7$ учимся на лучших 30%. Слишком маленький $q$ — учимся у посредственности; слишком большой — элита из двух эпизодов, и обучение превращается в лотерею. Обычно берут $q$ от 0.5 до 0.9.

### 13.5 Шаг 3: подстроить политику

Новая политика — просто **частоты действий в элитных эпизодах**:

$$
\pi_{\text{new}}(a \mid s) = \frac{\#\{(s, a) \text{ встретилось в элите}\}}{\#\{s \text{ встретилось в элите}\}} .
$$

Если состояние в элите ни разу не встретилось, оставляем для него старую строку политики. Разберём один шаг руками на игрушечном примере.

In [ ]:
# Игрушечный пример: 4 эпизода в среде с 2 состояниями и 2 действиями (0 и 1).
toy = [                       # (состояния, действия, return)
    ([0, 1], [0, 0], 0.0),
    ([0, 1], [1, 1], 1.0),
    ([0, 0], [0, 1], 0.0),
    ([0, 1], [1, 1], 1.0),
]
toy_returns = np.array([G for _, _, G in toy])
q = 0.5
threshold = np.quantile(toy_returns, q)
elite = [t for t in toy if t[2] >= threshold and t[2] > toy_returns.min()]
print(f"returns: {toy_returns},  порог (квантиль {q}) = {threshold},  в элите {len(elite)} эпизода")

policy_before = np.ones((2, 2)) / 2
counts = np.zeros((2, 2))
for s_list, a_list, _ in elite:
    for s, a in zip(s_list, a_list):
        counts[s, a] += 1
print("\nсколько раз элита делала действие a в состоянии s:\n", counts)

policy_after = policy_before.copy()
seen = counts.sum(axis=1) > 0
policy_after[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
print("\nполитика ДО :", policy_before.tolist())
print("политика ПОСЛЕ:", policy_after.tolist())

Вероятность действия 1 в состоянии 0 выросла с 0.5 до 1.0 — потому что так делали оба удачных эпизода. Никакой магии: посчитали частоты.

### 13.6 Почему «кросс-энтропия»

Шаг 3 выглядит как «подсчёт палочек», но за ним стоит оптимизационная задача. Мы хотим политику, при которой элитные действия наиболее вероятны, то есть максимизируем **логарифм правдоподобия** элитных пар $(s, a)$:

$$
\max_{\pi} \sum_{(s, a) \in \text{элита}} \log \pi(a \mid s)
\quad \Longleftrightarrow \quad
\min_{\pi} \ \underbrace{-\sum_{s} \sum_{a} p_{\text{элита}}(a \mid s) \log \pi(a \mid s)}_{\text{кросс-энтропия } H(p_{\text{элита}},\ \pi)} .
$$

Мы минимизируем **кросс-энтропию** между эмпирическим распределением действий в элитных эпизодах и нашей политикой — отсюда название. Для табличной политики эта задача решается в явном виде, и решение — те самые частоты, которые мы посчитали.

Отсюда главная мысль раздела и мостик к разделу 3:

> На каждой итерации CEM превращает задачу RL в обычную задачу **обучения с учителем**: «объекты» — состояния из элитных эпизодов, «правильные ответы» — действия, которые агент в них сделал, функция потерь — кросс-энтропия. Разница с supervised learning только в том, что разметку никто не давал: агент **сгенерировал её себе сам**, отобрав свои же лучшие попытки.

Это же объясняет, как метод масштабируется. Когда состояний слишком много, чтобы держать таблицу (кадры Atari, координаты робота), таблицу заменяют нейросетью $\pi_\theta(a \mid s)$, а шаг 3 — несколькими шагами градиентного спуска по той же самой кросс-энтропии (`nn.CrossEntropyLoss` в PyTorch, элитные состояния как батч, элитные действия как метки классов). Получается deep Cross-Entropy Method — им займёмся на неделе 5.

### 13.7 Алгоритм целиком

```text
policy = равномерная таблица (n_states × n_actions)
повторять n_iter раз:
    sessions  = [сыграть эпизод текущей policy] × n_sessions      # шаг 1
    threshold = квантиль уровня q от returns этих эпизодов        # шаг 2
    elite     = эпизоды с return >= threshold
    counts[s, a] = сколько раз элита делала a в s                 # шаг 3
    policy[s] = counts[s] / sum(counts[s])   для встреченных s
```

In [ ]:
def cross_entropy_method(env, n_iter=20, n_sessions=200, q=0.7, laplace=0.0, mix=1.0,
                         seed=0, max_steps=100):
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    log = {"mean": [], "threshold": [], "policies": [policy.copy()]}

    for it in range(n_iter):
        sessions = [run_session(env, policy, rng, max_steps) for _ in range(n_sessions)]   # шаг 1
        returns = np.array([G for _, _, G in sessions])

        threshold = np.quantile(returns, q)                                                # шаг 2
        # элита: не хуже порога и строго лучше самых плохих эпизодов
        elite = [s for s in sessions if s[2] >= threshold and s[2] > returns.min()]

        counts = np.full((n_states, n_actions), laplace)                                   # шаг 3
        for states, actions, _ in elite:
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy

        log["mean"].append(returns.mean())
        log["threshold"].append(threshold)
        log["policies"].append(policy.copy())
    return policy, log


policy_lake, log_lake = cross_entropy_method(env, n_iter=20, n_sessions=200, q=0.7)

plt.plot(log_lake["mean"], marker="o", label="средний return за итерацию")
plt.plot(log_lake["threshold"], marker=".", ls="--", label="порог элиты")
plt.xlabel("итерация"); plt.ylabel("return (здесь — доля успешных эпизодов)")
plt.title("Cross-Entropy на Frozen Lake 4×4"); plt.legend(); plt.show()

arrows = "←↓→↑"
desc = ["".join(c.decode() for c in row) for row in env.unwrapped.desc]
print("выученная политика (стрелка = самое вероятное действие):")
for r in range(4):
    print("  " + " ".join(desc[r][c] if desc[r][c] in "HG" else
                          arrows[int(np.argmax(policy_lake[r * 4 + c]))] for c in range(4)))

Готово: за одну-две итерации из равномерной политики получился маршрут до цели. Так быстро — потому что среда детерминированная: первая же удачная траектория попадает в элиту, и алгоритм её просто копирует. Это и удобно, и опасно, о чём сразу же ниже.

Ещё одно наблюдение к разделу 12: стрелки показывают лишь **самое вероятное** действие, а пользоваться такой политикой нужно именно случайно. В клетках, куда агент почти не заходил, вероятности остались равными, и «жадный» выбор действия там может зациклить агента между двумя клетками — с этим эффектом вы столкнётесь в домашнем задании.

### 13.8 Что может пойти не так

Алгоритм из трёх шагов, конечно, хрупкий. Три типичные болезни.

**1. Мало эпизодов на итерацию.** Если среди сыгранных эпизодов ни один не дошёл до цели, все returns одинаковы, отбирать нечего — политика не двигается, и так может продолжаться сколь угодно долго. Чем больше среда, тем реже случайный успех и тем больше эпизодов нужно на каждую итерацию.

**2. Схлопывание политики.** Частоты — очень жёсткое обновление: действие, не попавшее в элиту, получает вероятность ровно 0 и больше **никогда** не будет опробовано — агент сам закрыл себе доступ к этим данным (ровно то, о чём раздел 12). Одна удачная траектория может «выключить» все альтернативы, и алгоритм застрянет на первом найденном решении, даже если рядом есть куда лучше.

**3. Случайность среды.** На скользком льду эпизод может закончиться успехом просто потому, что повезло с направлением скольжения. В элиту тогда попадают везучие, а не умные, и политика учится на шуме.

Лекарство от болезни 2 (и отчасти от 3) — не доверять одной итерации полностью:

* **сглаживание по Лапласу**: к счётчикам добавляем $\lambda$, так что $\pi(a \mid s) = \dfrac{n(s,a) + \lambda}{n(s) + \lambda\,|\mathcal{A}|}$ — ни одно действие не обнуляется, агент продолжает изредка пробовать всё;
* **смешивание со старой политикой**: $\pi \leftarrow \alpha\,\pi_{\text{new}} + (1 - \alpha)\,\pi_{\text{old}}$ — политика меняется плавно, как learning rate в градиентном спуске.

Посмотрим на болезни 1 и 3 своими глазами.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (1) большое озеро 8x8: 20 эпизодов на итерацию против 250
big_env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
for n_sessions, color in [(20, "C3"), (250, "C0")]:
    for seed in range(3):
        _, log = cross_entropy_method(big_env, n_iter=25, n_sessions=n_sessions,
                                      q=0.7, seed=seed, max_steps=200)
        axes[0].plot(log["mean"], color=color, alpha=0.8,
                     label=f"{n_sessions} эпизодов на итерацию" if seed == 0 else None)
axes[0].set_title("Frozen Lake 8×8: по 3 запуска на каждый режим")

# (3) скользкий лёд: та же процедура, три запуска
slippery = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
for seed in range(3):
    _, log = cross_entropy_method(slippery, n_iter=30, n_sessions=250, q=0.7, seed=seed)
    axes[1].plot(log["mean"], color="C1", alpha=0.8, label="CEM" if seed == 0 else None)
axes[1].axhline(0.736, color="k", ls="--", lw=1, label="оптимальная политика: 74%")
axes[1].set_title("Frozen Lake 4×4 со скольжением")

for ax in axes:
    ax.set_xlabel("итерация"); ax.set_ylabel("доля успешных эпизодов"); ax.legend()
plt.tight_layout(); plt.show()

# схлопывание в числах: сколько вероятностей обнулилось за 3 итерации
for label, kwargs in [("без сглаживания", dict()),
                      ("Лаплас λ=1, смесь α=0.5", dict(laplace=1.0, mix=0.5))]:
    pol, _ = cross_entropy_method(env, n_iter=3, n_sessions=200, q=0.7, **kwargs)
    print(f"{label:24s}: действий с нулевой вероятностью {(pol == 0).mean():.0%}")

Слева: на большом озере два запуска из трёх с 20 эпизодами на итерацию так и не сдвинулись с нуля — успешных траекторий просто не нашлось, — а третьему повезло, и он обучился. С 250 эпизодами обучаются все три. Это не «неудачные гиперпараметры», а фундаментальное свойство метода: он живёт только за счёт случайных успехов.

Справа: на скользком льду кривые скачут и заметно не дотягивают до оптимальных 74% (больше в этой среде не может никто: скольжение уводит агента с маршрута). Причина — болезнь 3: элита набирается из везучих эпизодов. Настоящее лекарство — оценивать политику не по одному эпизоду, а по нескольким, и обновляться мягче; на семинаре вы попробуете это сами.

Схлопывание видно прямо в числах: без сглаживания уже через три итерации пятая часть всех вероятностей равна нулю — эти действия агент больше никогда не выберет. Со сглаживанием по Лапласу нулей нет вовсе.

### 13.9 Большое озеро: как политика становится маршрутом

Соберём всё вместе на поле 8×8 (64 состояния) со сглаживанием — теперь политика меняется плавно, и обучение можно посмотреть в динамике. Каждый кадр анимации — одна итерация: стрелка показывает самое вероятное действие, насыщенность — уверенность политики. Из полного хаоса за пару десятков итераций проступает маршрут в обход прорубей.

In [ ]:
big = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
policy_big, log_big = cross_entropy_method(big, n_iter=25, n_sessions=250, q=0.7,
                                           laplace=0.5, mix=0.5, max_steps=200)
plt.plot(log_big["mean"], marker=".")
plt.xlabel("итерация"); plt.ylabel("доля успешных эпизодов")
plt.title("Cross-Entropy на Frozen Lake 8×8 со сглаживанием"); plt.show()

def animate_policy(policies, desc, interval=350, width=4.2):
    nrow, ncol = len(desc), len(desc[0])
    fig, ax = plt.subplots(figsize=(width, width * 1.05))
    plt.close(fig)

    def draw(k):
        ax.clear(); ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlim(0, ncol); ax.set_ylim(nrow, 0)
        for r in range(nrow):
            for c in range(ncol):
                cell = desc[r][c]
                if cell in "HG":
                    ax.add_patch(plt.Rectangle((c, r), 1, 1,
                                               color="#a8c8ec" if cell == "H" else "#a9dfa9"))
                    ax.text(c + 0.5, r + 0.5, cell, ha="center", va="center", fontsize=10)
                    continue
                p = policies[k][r * ncol + c]
                a = int(np.argmax(p))
                ax.text(c + 0.5, r + 0.5, arrows[a], ha="center", va="center",
                        fontsize=14, alpha=0.15 + 0.85 * float(p[a]))
        for i in range(max(nrow, ncol) + 1):
            ax.axhline(i, color="k", lw=0.4); ax.axvline(i, color="k", lw=0.4)
        ax.set_title(f"итерация {k},  доля успехов "
                     f"{log_big['mean'][min(k, len(log_big['mean']) - 1)]:.0%}", fontsize=10)

    anim = animation.FuncAnimation(fig, draw, frames=len(policies), interval=interval)
    return HTML(anim.to_jshtml(default_mode="loop"))

desc_big = ["".join(c.decode() for c in row) for row in big.unwrapped.desc]
animate_policy(log_big["policies"], desc_big)

In [ ]:
# Эпизод выученной политикой, кадр за кадром. Действия выбираем случайно из pi(.|s) —
# именно так политика и была обучена.
rng = np.random.default_rng(1)
env_render = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False, render_mode="rgb_array")
obs, _ = env_render.reset(seed=0)
frames = [env_render.render()]
for _ in range(200):
    obs, r, terminated, truncated, _ = env_render.step(int(rng.choice(4, p=policy_big[obs])))
    frames.append(env_render.render())
    if terminated or truncated:
        break
env_render.close()
show_frames(frames, f"Выученная политика: цель за {len(frames)-1} шагов", interval=300, width=3.0)

### 13.10 Второй пример: тележка на горке

Frozen Lake — клеточный мир, где состояния уже пронумерованы. Что делать с непрерывным состоянием, как в Mountain Car (позиция и скорость)? Простейший ответ — **дискретизировать**: разбить каждую координату на несколько интервалов и считать номер клетки сетки номером состояния. Возьмём сетку $12 \times 12$, то есть 144 состояния и 3 действия.

Но сначала — честная демонстрация первой болезни во всей красе.

In [ ]:
NBINS = 12
LOW, HIGH = np.array([-1.2, -0.07]), np.array([0.6, 0.07])

def encode(obs):
    """Непрерывное состояние (x, v) -> номер клетки сетки."""
    i = np.clip(((obs - LOW) / (HIGH - LOW) * NBINS).astype(int), 0, NBINS - 1)
    return int(i[0] * NBINS + i[1])

def mc_session(env, policy, rng, max_steps=200):
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total, best_x = [], [], 0.0, obs[0]
    terminated = False
    for _ in range(max_steps):
        s = encode(obs)
        a = int(rng.choice(3, p=policy[s]))
        states.append(s); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r; best_x = max(best_x, obs[0])
        if terminated or truncated:
            break
    return states, actions, total, best_x, terminated

mc_env = gym.make("MountainCar-v0", max_episode_steps=200)
rng = np.random.default_rng(0)
uniform_mc = np.ones((NBINS * NBINS, 3)) / 3
sess = [mc_session(mc_env, uniform_mc, rng) for _ in range(200)]
returns_mc = np.array([s[2] for s in sess])
print(f"200 эпизодов равномерной политики: return от {returns_mc.min():.0f} до {returns_mc.max():.0f}, "
      f"уникальных значений: {len(np.unique(returns_mc))}")
print("порог элиты (квантиль 0.7):", np.quantile(returns_mc, 0.7))
print("=> все эпизоды одинаково плохи, отбирать нечего: CEM в таком виде не сдвинется с места.")

Это и есть **проблема разрежённой награды**: сигнал «−1 за шаг» одинаков для всех, пока ни один эпизод случайно не доехал до флажка, а случайной политикой это не происходит практически никогда.

Что можно сделать? Подсказать агенту промежуточную цель: отбирать элиту не по награде, а по **максимальной высоте**, которой достигла машинка. Это ровно тот приём, о котором предупреждал раздел 11: мы вмешиваемся в постановку задачи, и делать это надо осторожно (агент будет оптимизировать именно то, что мы написали). Зато сигнал становится плотным: «эта попытка заехала выше, чем та» — и алгоритм оживает.

In [ ]:
def cem_mountain_car(n_iter=45, n_sessions=150, q=0.7, laplace=1.0, mix=0.6, seed=0, max_steps=200):
    rng = np.random.default_rng(seed)
    env = gym.make("MountainCar-v0", max_episode_steps=max_steps)
    policy = np.ones((NBINS * NBINS, 3)) / 3
    log = {"height": [], "success": []}
    for it in range(n_iter):
        sessions = [mc_session(env, policy, rng, max_steps) for _ in range(n_sessions)]
        score = np.array([s[3] for s in sessions])          # элита — по максимальной высоте
        threshold = np.quantile(score, q)
        elite = [s for s in sessions if s[3] >= threshold]
        counts = np.full((NBINS * NBINS, 3), laplace)
        for states, actions, *_ in elite:
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = counts / counts.sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
        log["height"].append(score.mean())
        log["success"].append(np.mean([s[4] for s in sessions]))
    env.close()
    return policy, log

policy_mc, log_mc = cem_mountain_car()          # около полутора минут

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(log_mc["height"], color="C0", marker=".")
ax1.axhline(0.5, color="k", ls="--", lw=1)
ax1.set_xlabel("итерация"); ax1.set_ylabel("средняя максимальная высота x", color="C0")
ax2 = ax1.twinx()
ax2.plot(log_mc["success"], color="C2", marker=".")
ax2.set_ylabel("доля эпизодов, доехавших до флажка", color="C2")
plt.title("Cross-Entropy учит машинку раскачиваться"); plt.show()
print(f"доля успешных эпизодов на последней итерации: {log_mc['success'][-1]:.0%}")

In [ ]:
# Как выглядит выученная политика и как она едет.
grid = np.argmax(policy_mc.reshape(NBINS, NBINS, 3), axis=2)
plt.figure(figsize=(4.4, 4))
plt.imshow(grid.T, origin="lower", cmap="coolwarm",
           extent=[LOW[0], HIGH[0], LOW[1], HIGH[1]], aspect="auto")
plt.colorbar(ticks=[0, 1, 2], label="действие: 0 = газ влево, 1 = ничего, 2 = газ вправо")
plt.xlabel("позиция x"); plt.ylabel("скорость v")
plt.title("Выученная политика"); plt.show()

rng = np.random.default_rng(0)
mc_render = gym.make("MountainCar-v0", render_mode="rgb_array", max_episode_steps=200)
obs, _ = mc_render.reset(seed=3)
frames, t = [], 0
while True:
    if t % 4 == 0:
        frames.append(mc_render.render())
    a = int(rng.choice(3, p=policy_mc[encode(obs)]))
    obs, r, terminated, truncated, _ = mc_render.step(a)
    t += 1
    if terminated or truncated:
        break
mc_render.close()
show_frames(frames, f"Политика, выученная CEM: {'доехал' if terminated else 'не доехал'} за {t} шагов")

Обратите внимание на картинку политики: агент сам, без единой подсказки о физике, пришёл к правилу «газ в ту сторону, куда уже едешь» — то есть переоткрыл эвристику из раздела 9, которую там придумал человек. Красная область (газ вправо) — это положительные скорости, синяя (газ влево) — отрицательные.

### 13.11 Итог раздела

Что мы получили: рабочий алгоритм RL из трёх шагов — сыграть, отобрать лучшее, повторить то, что в нём делали. Никаких нейросетей, никакого знания среды, десяток строк кода.

Где у него границы:

* нужны **дискретные** состояния и действия (или дискретизация, как в Mountain Car, — а она плохо масштабируется: сетка $12^2$ ещё ничего, а $12^{17}$ для гуманоида уже невозможна);
* нужно **много эпизодов**: политика оценивается только целыми траекториями, информация об отдельных шагах не используется;
* плохо работает при **разрежённой награде** и в сильно **случайных** средах;
* учитывается только итог эпизода, поэтому длинный горизонт даётся тяжело.

Куда двигаться дальше:

* **неделя 5** — та же схема, но политика-таблица заменяется нейросетью, а шаг 3 — градиентным спуском по кросс-энтропии: тот же алгоритм решает CartPole и LunarLander;
* **недели 3–4** — методы, которые используют структуру MDP и уравнение Беллмана из раздела 10: они извлекают информацию из **каждого шага**, а не из целого эпизода, и потому куда экономнее по данным.

## 14. Карта курса

![course](../../assets/course_map.png)

Первые четыре недели — **основы**: постановка задачи, построение сред, табличные value-based и policy-based методы. Дальше те же идеи переносим на нейросети (**deep RL**: DQN, policy gradient, actor-critic), потом продвинутые методы (PPO, TD3, model-based), и расширения: иерархический и многоагентный RL, трансформеры.

![taxonomy](../../assets/rl_taxonomy.png)

Всё, что в этой таблице, мы **реализуем сами** с нуля: от Cross-Entropy до PPO и TD3. RL-алгоритмы печально известны тем, что «почти работающая» реализация не работает вовсе, и понять, где ошибка, можно только зная каждую строчку.

## Итоги лекции

1. **Reinforcement Learning зарождался в психологии.** «Подкрепление» и закон эффекта описывают тот же механизм, который мы теперь программируем; математику к нему добавила теория управления.
2. **Reinforcement Learning можно выделить в отдельную группу алгоритмов машинного обучения.** Нет правильных ответов, награда отложена, данные порождает сам агент, нужно исследовать. Ни одной из этих черт нет в supervised и unsupervised learning.
3. **Reinforcement Learning широко применяется в самых разных областях: от игр до управления охлаждением дата-центров и от торговли на бирже до роевого поведения дронов.** Рецепт один: среда + числовая цель.

Словарь, который нужно унести с собой: **агент, среда, состояние, действие, награда, политика, эпизод, терминальное состояние, return, марковское свойство, MDP, ценность $V$ и $Q$, уравнение Беллмана, exploration и exploitation**.

## Литература

* R. Sutton, A. Barto. *Reinforcement Learning: An Introduction*, 2nd ed. — [бесплатный PDF](http://incompleteideas.net/book/the-book-2nd.html). Главы 1–3 покрывают сегодняшнюю лекцию.
* D. Silver. [UCL Course on RL](https://www.davidsilver.uk/teaching/), лекции 1–2: введение и MDP.
* А. Плаксин. [Курс imm-rl-lab](https://github.com/imm-rl-lab/reinforcement_learning_course), лекция 1: введение и метод Cross-Entropy.
* [OpenAI Spinning Up](https://spinningup.openai.com/) — короткое введение + чистые реализации.
* [Gymnasium](https://gymnasium.farama.org/) — документация по средам и интерфейсу.

## На семинаре и дома

* Семинар (`../seminar/seminar.ipynb`): интерфейс Gymnasium на Frozen Lake, политика-таблица, среда бандита и агенты ε-greedy / UCB1, реализация Cross-Entropy.
* Мини-семинар по PyTorch (`../seminar/pytorch_intro.ipynb`) для тех, кто с ним не работал.
* ДЗ (`../homework/homework.ipynb`): бандиты, теория (return, вывод Беллмана, марковская цепь, MDP на бумаге), Cross-Entropy на Frozen Lake 8×8.